In [1]:
import os, requests, json, pyodbc, logging, time
import pandas as pd
from datetime import datetime, timezone
from dotenv import load_dotenv
from sqlalchemy import create_engine
from tqdm import tqdm
from datetime import datetime, timedelta
import unicodedata
import numpy as np


load_dotenv()

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

API_KEY = os.getenv("INAGENT_API_KEY")
ENDPOINT = os.getenv("INAGENT_URL")

CREW_MAPPING = {
    os.getenv("INAGENT_CREW_ID"): "DENTAL",
    os.getenv("INAGENT_CREW_ID2"): "OMV"
}
CREW_IDS = list(CREW_MAPPING.keys())


In [2]:
hoy = datetime.now()
ayer = hoy - timedelta(days=1)

env_start = os.getenv("START_DATE")
env_end = os.getenv("END_DATE")

if env_start:
    start_date = env_start
else:
    start_date = ayer.strftime("%Y-%m-%dT00:00:00")

if env_end:
    end_date = env_end
else:
    end_date = ayer.strftime("%Y-%m-%dT23:59:59")

print(f"Extrayendo datos desde {start_date} hasta {end_date}")


Extrayendo datos desde 2026-03-15T00:00:00 hasta 2026-04-06T23:59:59


In [3]:
def to_unix_ms(iso_date):
    dt = datetime.fromisoformat(iso_date).replace(tzinfo=timezone.utc)
    return int(dt.timestamp() * 1000)

def safe_json_parse(val):
    try:
        return json.loads(val) if (val and val != 'null') else {}
    except:
        return {}

print(" Herramientas listas: to_unix_ms y safe_json_parse.")

 Herramientas listas: to_unix_ms y safe_json_parse.


In [4]:
all_data = []
page_size = 100 # Límite recomendado por la API [cite: 141]

for crew in CREW_IDS:
    page = 0
    label_origen = CREW_MAPPING.get(crew, "DESCONOCIDO")
    logger.info(f"Extrayendo datos de {label_origen} (ID: {crew})")
    
    while True:
        params = {
            "crew_id": crew,
            "start_ts": to_unix_ms(start_date),
            "end_ts": to_unix_ms(end_date),
            "page": page,
            "pageSize": page_size
        }
        
        headers = {"apikey": API_KEY}
        res = requests.get(ENDPOINT, headers=headers, params=params)
        
        if res.status_code != 200:
            logger.error(f"Fallo en {label_origen}, Página {page}: {res.text}")
            break
            
        data_payload = res.json().get("data", {})
        rows = data_payload.get("rows", [])
        
        if page == 0 and crew == CREW_IDS[0]:
            cols = data_payload.get("dataSchema", {}).get("columnNames", [])
            if "Origen" not in cols:
                cols.append("Origen")
        
        if not rows:
            break
            
        for row in rows:
            row.append(label_origen)
            
        all_data.extend(rows)
        logger.info(f"Página {page} de {label_origen} lista. Total: {len(all_data)}")
        
        if len(rows) < page_size: # Fin de datos para este equipo [cite: 262]
            break
        page += 1

df_raw = pd.DataFrame(all_data, columns=cols)


2026-04-07 11:58:03,178 - INFO - Extrayendo datos de DENTAL (ID: 766cddc2-eaf9-464e-8f6d-8854ef927ff3)
2026-04-07 11:58:04,432 - INFO - Página 0 de DENTAL lista. Total: 49
2026-04-07 11:58:04,432 - INFO - Extrayendo datos de OMV (ID: 2539ab63-c408-446a-b13e-1eef4f7c1ba3)
2026-04-07 11:58:05,510 - INFO - Página 0 de OMV lista. Total: 66


In [5]:
native_whitelist = [
    'Id', 
    'Id Externo',
    'Id Canal',
    'Canal',
    'Timestamp',
    'Inicio',
    'Fin',
    'Duración (s)', 
    'Análisis Sentimental',
    'Tema general de la conversación',
    #'Resumen',
    'Fue resuelta',
    'Fue solo agradecimiento',
    'Herramientas Usadas',
    'Es saliente',
    'Fue abandonada',
    'Contexto',
    'Origen'
]

In [6]:
context_whitelist = [
    'toolLogs',
    'tarjeta',
    'proxyData_sip_attributes_sip_trunkPhoneNumber',
    'proxyData_sip_attributes_sip_h_x-tarjeta-id'
]

In [7]:
df_native = df_raw[[c for c in native_whitelist if c in df_raw.columns]].copy()

ctx_raw = pd.json_normalize(df_raw['Contexto'].apply(safe_json_parse))
ctx_raw.columns = [c.replace(".", "_") for c in ctx_raw.columns]

In [8]:
ctx_selected = ctx_raw[[c for c in context_whitelist if c in ctx_raw.columns]].add_prefix('ctx_')
df_base = pd.concat([df_native, ctx_selected], axis=1)
print(f"{df_base.shape}")

(66, 21)


In [9]:
col_logs = 'ctx_toolLogs'

df_tools_exploded = df_base[['Id', col_logs]].dropna(subset=[col_logs]).explode(col_logs)
tool_rows = df_tools_exploded[col_logs].apply(lambda x: x if isinstance(x, (dict, list)) else safe_json_parse(x)).tolist()
df_tools_flat = pd.json_normalize(tool_rows)

In [10]:
df_tools_flat.to_csv("tools.csv",index=False)

In [11]:
tool_whitelist = [
    'URL_fetch', 'body_fetch', 'return_fetch', 'tool', 'status', 
    'timestamp', 'code_fetch','return_fetch.msg',
    'return_fetch.message','return_fetch.data.client_complete_name',
    'return_fetch.data.policy_number','return_fetch.data.program_name',
    'return_fetch.data.program_status',
    'return_fetch.data.client_account',
    'return_fetch.data.client_telefono',
    'return_fetch.success',
    'return_fetch.data.client_card',
    'body_fetch.cas',
    'body_fetch.cancelMotive',
    'return_fetch.response.cas_folio',
    'body_fetch.scheduleDate', #date
    'body_fetch.specialty',
    'return_fetch.response.nameDoctor',
    'return_fetch.response.name_service',
    'return_fetch.response.creationDate' #Timestrap
    'return_fetch.response.title',
    'return_fetch.response.provider',
    'return_fetch.response.Kinship',
    'return_fetch.response.category',
    'return_fetch.response.nameDoctor',
]

In [12]:
cols_t = [c for c in tool_whitelist if c in df_tools_flat.columns]
df_tools_final = df_tools_flat[cols_t].copy()
df_tools_final.columns = [f"tool_{c.replace('.', '_')}" for c in df_tools_final.columns]

df_tools_final.index = df_tools_exploded.index
df_tools_merged = pd.concat([df_tools_exploded[['Id']], df_tools_final], axis=1)

print(f'{df_tools_merged.shape}')

(80, 29)


In [13]:
df_tools_final.info()

<class 'pandas.DataFrame'>
Index: 80 entries, 0 to 63
Data columns (total 28 columns):
 #   Column                                       Non-Null Count  Dtype  
---  ------                                       --------------  -----  
 0   tool_URL_fetch                               80 non-null     str    
 1   tool_body_fetch                              13 non-null     str    
 2   tool_return_fetch                            15 non-null     str    
 3   tool_tool                                    80 non-null     str    
 4   tool_status                                  80 non-null     str    
 5   tool_timestamp                               80 non-null     str    
 6   tool_code_fetch                              53 non-null     float64
 7   tool_return_fetch_msg                        15 non-null     str    
 8   tool_return_fetch_message                    24 non-null     str    
 9   tool_return_fetch_data_client_complete_name  15 non-null     str    
 10  tool_return_fetch_da

In [14]:
df_final = df_base.merge(df_tools_merged, on='Id', how='left')
df_final.columns = [c.replace(" ", "_").replace("(", "").replace(")", "").replace(".", "_") for c in df_final.columns]

In [15]:
def aplicar_ancla_maestra(df): 
    df['tool_timestamp_dt'] = pd.to_datetime(df['tool_timestamp'], errors='coerce')
    df = df.sort_values(by=['Id', 'tool_timestamp_dt'], ascending=[True, False])
    
    es_el_ancla = ~df.duplicated(subset=['Id'], keep='first')
    
    df['interaccion_unica'] = np.where(es_el_ancla,1,0)
    df['Estatus_Final'] = np.where(es_el_ancla, df['tool_tool'], None)
    
    return df

df_preparado = aplicar_ancla_maestra(df_final)

In [16]:
df_preparado.info()

<class 'pandas.DataFrame'>
Index: 122 entries, 34 to 115
Data columns (total 52 columns):
 #   Column                                             Non-Null Count  Dtype              
---  ------                                             --------------  -----              
 0   Id                                                 122 non-null    str                
 1   Id_Externo                                         122 non-null    str                
 2   Id_Canal                                           20 non-null     str                
 3   Canal                                              122 non-null    str                
 4   Timestamp                                          122 non-null    str                
 5   Inicio                                             122 non-null    str                
 6   Fin                                                122 non-null    str                
 7   Duración_s                                         122 non-null    int64     

In [17]:
def crear_id_compuesto_pro(df):
    logger.info(" Generando identificadores únicos...")
    

    df['tool_tool'] = df['tool_tool'].fillna('SIN_HERRAMIENTA') # <--- CLAVE
    
    tool = df['tool_timestamp'].astype(str)
    
    df['id_registro'] = (
        df['Id'].astype(str) + "_" + 
        df['tool_tool'].astype(str) + "_" + 
        tool
    )
    return df

In [18]:
crear_id_compuesto_pro(df_preparado)

2026-04-07 11:58:05,623 - INFO -  Generando identificadores únicos...


,Id,Id_Externo,Id_Canal,Canal,Timestamp,Inicio,Fin,Duración_s,Análisis_Sentimental,Tema_general_de_la_conversación,...,tool_return_fetch_response_nameDoctor,tool_return_fetch_response_name_service,tool_return_fetch_response_provider,tool_return_fetch_response_Kinship,tool_return_fetch_response_category,tool_return_fetch_response_nameDoctor,tool_timestamp_dt,interaccion_unica,Estatus_Final,id_registro
34,0dc649a2-7ca0-4a99-81ca-3c87095400e3,,NaN,,1774044871000,2026-03-20T22:14:31.000Z,2026-03-20T22:15:34.454Z,63,neutral,agendar cita,...,NaN,NaN,NaN,NaN,NaN,NaN,2026-03-20 22:15:07.500000+00:00,1,get_titular,0dc649a2-7ca0-4a99-81ca-3c87095400e3_get_titul...
33,0dc649a2-7ca0-4a99-81ca-3c87095400e3,,NaN,,1774044871000,2026-03-20T22:14:31.000Z,2026-03-20T22:15:34.454Z,63,neutral,agendar cita,...,NaN,NaN,NaN,NaN,NaN,NaN,2026-03-20 22:14:54.368000+00:00,0,NaN,0dc649a2-7ca0-4a99-81ca-3c87095400e3_inicializ...
32,0dc649a2-7ca0-4a99-81ca-3c87095400e3,,NaN,,1774044871000,2026-03-20T22:14:31.000Z,2026-03-20T22:15:34.454Z,63,neutral,agendar cita,...,NaN,NaN,NaN,NaN,NaN,NaN,2026-03-20 22:14:33.440000+00:00,0,NaN,0dc649a2-7ca0-4a99-81ca-3c87095400e3_inicializ...
67,0e05eb7a-0b51-4f95-9e00-51bba33a72f7,,d6aef35f-bbaf-44c9-b960-b31cff41b875,Livekit_SipCall,1773859228000,2026-03-18T18:40:28.000Z,2026-03-18T19:01:48.062Z,1280,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaT,1,NaN,NaN
74,0f361660-3f40-4150-9ba9-d65ed171396a,,d6aef35f-bbaf-44c9-b960-b31cff41b875,Livekit_SipCall,1773858315000,2026-03-18T18:25:15.000Z,2026-03-18T18:26:36.421Z,81,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaT,1,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
119,fb3cc062-8a73-4c81-9a5c-fee073f905a4,,NaN,,1774289801000,2026-03-23T18:16:41.000Z,2026-03-23T18:19:45.106Z,184,neutral,Cita nutrición,...,Sergio Moreno Sosa,Dr. Sergio Moreno,liverpool,Beneficiario,Nutrición,Sergio Moreno Sosa,2026-03-23 18:19:12.435000+00:00,1,set_checkup_OMV,fb3cc062-8a73-4c81-9a5c-fee073f905a4_set_check...
118,fb3cc062-8a73-4c81-9a5c-fee073f905a4,,NaN,,1774289801000,2026-03-23T18:16:41.000Z,2026-03-23T18:19:45.106Z,184,neutral,Cita nutrición,...,NaN,NaN,NaN,NaN,NaN,NaN,2026-03-23 18:18:30.036000+00:00,0,NaN,fb3cc062-8a73-4c81-9a5c-fee073f905a4_get_horar...
117,fb3cc062-8a73-4c81-9a5c-fee073f905a4,,NaN,,1774289801000,2026-03-23T18:16:41.000Z,2026-03-23T18:19:45.106Z,184,neutral,Cita nutrición,...,NaN,NaN,NaN,NaN,NaN,NaN,2026-03-23 18:18:28.794000+00:00,0,NaN,fb3cc062-8a73-4c81-9a5c-fee073f905a4_get_fecha...
116,fb3cc062-8a73-4c81-9a5c-fee073f905a4,,NaN,,1774289801000,2026-03-23T18:16:41.000Z,2026-03-23T18:19:45.106Z,184,neutral,Cita nutrición,...,NaN,NaN,NaN,NaN,NaN,NaN,2026-03-23 18:17:45.844000+00:00,0,NaN,fb3cc062-8a73-4c81-9a5c-fee073f905a4_get_benef...


In [19]:
df_preparado.to_csv("prepare.csv",index=False)

In [20]:
columnas_sql_reales = [
    'Id', 
    'Id_Externo', 
    'Id_Canal', 
    'Canal', 
    'Timestamp', 
    'Inicio', 
    'Fin', 
    'Duración_s', 
    'Análisis_Sentimental', 
    'Tema_general_de_la_conversación', 
    'Fue_resuelta', 
    'Fue_solo_agradecimiento', 
    'Herramientas_Usadas', 
    'Es_saliente', 
    'Fue_abandonada', 
    'Origen', 
    'ctx_tarjeta', 
    'ctx_proxyData_sip_attributes_sip_trunkPhoneNumber', 
    'ctx_proxyData_sip_attributes_sip_h_x-tarjeta-id', 
    'tool_URL_fetch', 
    'tool_body_fetch', 
    'tool_return_fetch', 
    'tool_tool', 
    'tool_status', 
    'tool_timestamp', 
    'tool_code_fetch', 
    'tool_return_fetch_msg', 
    'tool_return_fetch_message', 
    'tool_return_fetch_data_client_complete_name', 
    'tool_return_fetch_data_policy_number', 
    'tool_return_fetch_data_program_name', 
    'tool_return_fetch_data_program_status', 
    'tool_return_fetch_data_client_account', 
    'tool_return_fetch_data_client_telefono', 
    'tool_return_fetch_success', 
    'tool_return_fetch_data_client_card', 
    'tool_body_fetch_cas', 
    'tool_body_fetch_cancelMotive', 
    'tool_return_fetch_response_cas_folio', 
    'tool_body_fetch_scheduleDate', 
    'tool_body_fetch_specialty', 
    'tool_return_fetch_response_nameDoctor', 
    'tool_return_fetch_response_name_service', 
    'tool_return_fetch_response_provider', 
    'tool_return_fetch_response_Kinship', 
    'tool_return_fetch_response_category', 
    'tool_timestamp_dt', #Fecha
    'interaccion_unica', #Bin 1 y 0
    'Estatus_Final', #STR Varchar 
    'id_registro'
]

In [21]:
def pipeline_maestro_final(df, whitelist):
    df_sql = df.copy()
    
    def limpiar_nombres(txt):
        if not isinstance(txt, str): return txt
        txt = "".join(c for c in unicodedata.normalize('NFD', txt) if unicodedata.category(c) != 'Mn')
        return txt.replace(" ", "_").replace("(", "").replace(")", "").replace(".", "_")

    df_sql.columns = [limpiar_nombres(c) for c in df_sql.columns]
    whitelist_limpia = [limpiar_nombres(c) for c in whitelist]

    if 'Origen' in df_sql.columns:
        df_sql['Id_Canal'] = df_sql['Origen']
        df_sql['Id_Externo'] = df_sql['id_registro'] # PK técnica de fila
        
        if 'interaccion_unica' in df_sql.columns:
            df_sql['tool_return_fetch_response_costPIFDoctor'] = df_sql['interaccion_unica']
        
        if 'Estatus_Final' in df_sql.columns:
            df_sql['tool_return_fetch_response_typeOfService'] = df_sql['Estatus_Final']

    df_sql = df_sql[[c for c in whitelist_limpia if c in df_sql.columns]]

    cols_num = [
        'Duracion_s', 'Herramientas_Usadas', 'tool_code_fetch',
        'tool_return_fetch_httpCode', 'tool_return_fetch_response_costPIFDoctor',
        'tool_return_fetch_response_selected_dentist'
    ]
    
    cols_date = ['Inicio', 'Fin', 'tool_timestamp_dt']

    for col in df_sql.columns:
        
        if col in cols_date:
            df_sql[col] = pd.to_datetime(df_sql[col], errors='coerce')
            df_sql[col] = df_sql[col].astype(object).where(pd.notnull(df_sql[col]), None)
            
        elif col in cols_num:
            df_sql[col] = pd.to_numeric(df_sql[col], errors='coerce')
            df_sql[col] = df_sql[col].astype(object).where(pd.notnull(df_sql[col]), None)
            
        elif any(x in col for x in ['Fue_', 'Es_', 'Cerrada_']):
            df_sql[col] = pd.to_numeric(df_sql[col], errors='coerce').fillna(0).astype(int)
            
        else:
            df_sql[col] = df_sql[col].astype(str).replace(['n.n','nan', 'None', 'NaN', 'null'], None)
            df_sql[col] = df_sql[col].where(df_sql[col].notnull(), None)

    return df_sql

In [22]:
df_listo = pipeline_maestro_final(df_preparado, columnas_sql_reales)

In [23]:
df_listo.rename(columns={
        'ctx_proxyData_sip_attributes_sip_h_x-tarjeta-id': 'ctx_proxyData_sip_attributes_sip_h_x_tarjeta_id'
    }, inplace=True)

In [24]:
para_SQL = [
    'Id', 
    'Id_Externo', 
    'Id_Canal', 
    'Canal', 
    'Timestamp', 
    'Inicio', 
    'Fin', 
    'Duracion_s', 
    'Analisis_Sentimental', 
    'Tema_general_de_la_conversacion', 
    'Fue_resuelta', 
    'Fue_solo_agradecimiento', 
    'Herramientas_Usadas', 
    'Es_saliente', 
    'Fue_abandonada', 
    'Origen', 
    'ctx_tarjeta', 
    'ctx_proxyData_sip_attributes_sip_trunkPhoneNumber', 
    'ctx_proxyData_sip_attributes_sip_h_x_tarjeta_id', 
    'tool_URL_fetch', 
    'tool_body_fetch', 
    'tool_return_fetch', 
    'tool_tool', 
    'tool_status', 
    'tool_timestamp', 
    'tool_code_fetch', 
    'tool_return_fetch_msg', 
    'tool_return_fetch_message', 
    'tool_return_fetch_data_client_complete_name', 
    'tool_return_fetch_data_policy_number', 
    'tool_return_fetch_data_program_name', 
    'tool_return_fetch_data_program_status', 
    'tool_return_fetch_data_client_account', 
    'tool_return_fetch_data_client_telefono', 
    'tool_return_fetch_success', 
    'tool_return_fetch_data_client_card', 
    'tool_body_fetch_cas', 
    'tool_body_fetch_cancelMotive', 
    'tool_return_fetch_response_cas_folio', 
    'tool_body_fetch_scheduleDate', 
    'tool_body_fetch_specialty', 
    'tool_return_fetch_response_nameDoctor', 
    'tool_return_fetch_response_name_service', 
    'tool_return_fetch_response_provider', 
    'tool_return_fetch_response_Kinship', 
    'tool_return_fetch_response_category', 
    'tool_timestamp_dt', 
    'interaccion_unica', 
    'Estatus_Final',
   # 'id_registro' 
]

In [25]:
df_produccion = df_listo[[c for c in para_SQL if c in df_listo.columns]].copy()

df_produccion = df_produccion.loc[:, ~df_produccion.columns.duplicated()].copy()

duplicadas = df_produccion.columns[df_produccion.columns.duplicated()].tolist()
logger.info(f"Columnas duplicadas aniquiladas: {duplicadas}")

2026-04-07 11:58:05,709 - INFO - Columnas duplicadas aniquiladas: []


In [26]:
df_para_sql = df_produccion.copy()

TABLE_NAME = "dbo.inagent" #

conn_str = (
    f"DRIVER={{ODBC Driver 17 for SQL Server}};"
    f"SERVER={os.getenv('DB_SERVER')},{os.getenv('DB_PORT')};"
    f"DATABASE={os.getenv('BD')};"
    f"UID={os.getenv('DB_USER')};"
    f"PWD={os.getenv('DB_PASS')}"
)

try:
    conn = pyodbc.connect(conn_str)
    cursor = conn.cursor()
    cursor.fast_executemany = True 

    cursor.execute(f"IF OBJECT_ID('tempdb..#stg_inagent') IS NOT NULL DROP TABLE #stg_inagent")
    
    cols = df_para_sql.columns.tolist()
    col_names_bracketed = ", ".join(f"[{c}]" for c in cols)
    
    cursor.execute(f"SELECT TOP 0 {col_names_bracketed} INTO #stg_inagent FROM {TABLE_NAME}") 

    placeholders = ", ".join("?" for _ in cols)
    sql_insert = f"INSERT INTO #stg_inagent ({col_names_bracketed}) VALUES ({placeholders})"
    
    data_to_load = [tuple(x) for x in df_para_sql.values]
    
    logger.info(f" Subiendo {len(data_to_load)} registros a Staging...")
    cursor.executemany(sql_insert, data_to_load)
    
    sql_merge = f"""
    MERGE {TABLE_NAME} AS target
    USING #stg_inagent AS source
    ON (target.Id_Externo= source.Id_Externo)
    WHEN MATCHED THEN
        UPDATE SET 
            target.Analisis_Sentimental = source.Analisis_Sentimental,
            target.Tema_general_de_la_conversacion = source.Tema_general_de_la_conversacion,
            target.tool_status = source.tool_status,
            target.tool_return_fetch_message = source.tool_return_fetch_message
    WHEN NOT MATCHED THEN
        INSERT ({col_names_bracketed})
        VALUES ({', '.join(f'source.[{c}]' for c in cols)});
    """
    
    logger.info("Ejecutando MERGE en tabla definitiva...")
    cursor.execute(sql_merge)
    conn.commit()
    logger.info(f"ÉXITO: {len(df_para_sql)} registros sincronizados correctamente.")

except Exception as e:
    if 'conn' in locals(): conn.rollback()
    logger.error(f"Error en SQL: {e}")
finally:
    if 'cursor' in locals(): cursor.close()
    if 'conn' in locals(): conn.close()

2026-04-07 11:58:07,561 - INFO -  Subiendo 122 registros a Staging...
2026-04-07 11:58:23,339 - INFO - Ejecutando MERGE en tabla definitiva...
2026-04-07 11:58:23,611 - ERROR - Error en SQL: ('42000', '[42000] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]The MERGE statement attempted to UPDATE or DELETE the same row more than once. This happens when a target row matches more than one source row. A MERGE statement cannot UPDATE/DELETE the same row of the target table multiple times. Refine the ON clause to ensure a target row matches at most one source row, or use the GROUP BY clause to group the source rows. (8672) (SQLExecDirectW)')


In [27]:
tools_master_whitelist = [
    'inicializar_sesion',           # Obtiene información del cliente [cite: 47]
    'CerrarConversacionPorUsuario', # Cierre por decisión del usuario [cite: 48]
    'CerrarSesionPorTimeout',       # Cierre por inactividad [cite: 49]
    'get_fecha',                    # Obtiene fecha actual [cite: 50]
    'upsert_beneficiario',          # Registra/actualiza beneficiario [cite: 51]
    'get_beneficiarios',            # Obtiene lista de beneficiarios [cite: 52]
    'get_servicios_siniestralidad', # Consulta de servicios [cite: 53]
    'validar_tarjeta',              # Consulta beneficios [cite: 54]
    'TransferenciaAsesor',          # Envío a agente humano [cite: 113] 
    'get_titular',                  # Obtiene ID del titular [cite: 55]
    'get_horarios',                 # Consulta disponibilidad [cite: 56]
    'cancelar_cita',                # Cancela cita agendada [cite: 57]
    'set_checkup',                  # Crea nueva cita [cite: 58]
    'set_context'                   # Crea contexto con Talkdesk [cite: 59]
]